# Collision Anticipation: Data and Frame Features

<div dir="rtl">

**هدف.** مسئله‌ی ۱ می‌پرسد «در این ویدئو تصادف شد یا نه». این کار بعد از حادثه است. مسئله‌ی جدید یک قدم جلوتر است: «تا یکی دو ثانیه‌ی دیگر تصادف می‌شود یا نه». مدل در هر لحظه فقط فریم‌های قبل از آن لحظه را می‌بیند و یک امتیاز خطر می‌دهد. این نوت‌بوک داده را آماده می‌کند. مدل در نوت‌بوک بعدی ساخته می‌شود.

**داده.** دیتاست Nexar روی HuggingFace:
- ۱۵۰۰ ویدئوی train: ۷۵۰ تصادف یا نزدیک به تصادف، ۷۵۰ رانندگی عادی.
- ۱۳۴۴ کلیپ آزمون رسمی در `test-public` و `test-private`. کلیپ‌های مثبت طوری بریده شده‌اند که ۰.۵، ۱ یا ۱.۵ ثانیه قبل از حادثه تمام شوند، پس دقیقاً همین مسئله را می‌سنجند.

برچسب‌های زمانی از متادیتا می‌آیند: `time_of_event` لحظه‌ی حادثه است و `time_of_alert` اولین لحظه‌ای که یک راننده‌ی دقیق می‌فهمد خطری در راه است.

**مراحل.**
1. دانلود و بررسی متادیتا: زمان حادثه، فاصله‌ی هشدار تا حادثه، شرایط محیطی.
2. نمونه‌برداری ۱۰ فریم در ثانیه با اندازه‌ی 392×224.
3. ساخت بردار ویژگی هر فریم با DINOv2-small یخ‌زده.
4. ذخیره‌ی تکه‌تکه‌ی ویژگی‌ها در Drive، تا اجرا بعد از قطع اتصال از همان‌جا ادامه پیدا کند.
5. بررسی خروجی‌ها.

**خروجی‌ها** در `MyDrive/project2/anticipation/`:
- `features/`: ویژگی و زمان هر فریم برای همه‌ی ویدئوها
- `data/videos.csv`: متادیتا به‌علاوه‌ی مشخصات فریم‌های هر ویدئو
- `data/`: فایل‌های ارزیابی رسمی دیتاست
- `sample_videos/`: هشت ویدئوی کم‌حجم برای داشبورد

**محیط.** Google Colab با GPU. دیتاست gated است: دانلود با توکن یک حساب HuggingFace انجام می‌شود که شرایط دیتاست را پذیرفته باشد.

</div>

## Setup

<div dir="rtl">

ویدئوها روی دیسک محلی کولب دانلود و بعد از پردازش پاک می‌شوند. فقط ویژگی‌ها در Drive می‌مانند، چون حجمشان خیلی کمتر از ویدئوهاست (حدود ۲.۲ گیگ، در برابر ۳۱ گیگ کل دیتاست روی HuggingFace).

</div>

In [1]:
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import torch
from huggingface_hub import snapshot_download
from huggingface_hub.utils import disable_progress_bars
from plotly.subplots import make_subplots
from transformers import AutoModel

disable_progress_bars()
pio.templates.default = "plotly_white"

In [2]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/project2/anticipation")
DATA_DIR = DRIVE_DIR / "data"
FEATURE_DIR = DRIVE_DIR / "features"
SAMPLE_DIR = DRIVE_DIR / "sample_videos"
LOCAL_DIR = Path("/content/nexar")
for folder in [DATA_DIR, FEATURE_DIR, SAMPLE_DIR, LOCAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if device == "cuda" else "cpu")
if device == "cpu":
    raise RuntimeError("A GPU runtime is needed for feature extraction.")

drive_free = shutil.disk_usage(DRIVE_DIR).free / 1e9
local_free = shutil.disk_usage(LOCAL_DIR).free / 1e9
print(f"Free space | Drive: {drive_free:.1f} GB | local disk: {local_free:.1f} GB")
if drive_free < 3:
    print("Warning: about 2.5 GB of features will be written to Drive.")

Mounted at /content/drive
Device: Tesla T4
Free space | Drive: 1.8 GB | local disk: 120.8 GB


In [3]:
REPO_ID = "nexar-ai/nexar_collision_prediction"
REVISION = "7535d0656dac31d7da2846913bb69c6331d2a70a"
FOLDERS = ["train/positive", "train/negative",
           "test-public/positive", "test-public/negative",
           "test-private/positive", "test-private/negative"]

FPS_OUT = 10          # frames kept per second
FRAME_W, FRAME_H = 392, 224
POST_EVENT = 2.0      # seconds kept after the event in train positives
CHUNK_SIZE = 50       # videos per saved feature file
ENCODER_NAME = "facebook/dinov2-small"
SEED = 42

## Dataset Files

<div dir="rtl">

هر کدام از سه پوشه‌ی داده دو زیرپوشه‌ی `positive` و `negative` دارد و هر زیرپوشه یک `metadata.csv`. نسخه‌ی دیتاست با شناسه‌ی commit ثابت شده تا نتیجه‌ها تکرارپذیر باشند. فایل‌های ارزیابی رسمی هم کنار متادیتا دانلود و در Drive ذخیره می‌شوند.

توکن HuggingFace از Colab Secrets با نام `HF_TOKEN` خوانده می‌شود. اگر آنجا نباشد، با یک کادر مخفی پرسیده می‌شود. توکن چاپ یا ذخیره نمی‌شود.

</div>

In [4]:
from getpass import getpass
from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass("HuggingFace token: ").strip()


def download(patterns):
    for attempt in range(3):
        try:
            return snapshot_download(REPO_ID, repo_type="dataset", revision=REVISION, token=HF_TOKEN,
                                     allow_patterns=patterns, local_dir=LOCAL_DIR)
        except Exception as error:
            message = str(error)
            if "401" in message or "403" in message or "gated" in message.lower():
                raise RuntimeError("No access to the dataset. The token must belong to an account "
                                   "that has accepted the dataset terms on HuggingFace.") from None
            print(f"Download failed (attempt {attempt + 1}): {message}")
            time.sleep(20)
    raise RuntimeError("Download failed three times.")


ROOT_FILES = ["README.md", "evaluate_submission.py", "sample_submission.csv",
              "solution.csv", "time_to_accident_test_map.csv"]
download(ROOT_FILES + [f"{folder}/metadata.csv" for folder in FOLDERS])
for name in ROOT_FILES:
    shutil.copy(LOCAL_DIR / name, DATA_DIR / name)

tables = []
for folder in FOLDERS:
    table = pd.read_csv(LOCAL_DIR / folder / "metadata.csv", dtype={"file_name": str})
    split, group = folder.split("/")
    table.insert(0, "split", split)
    table.insert(1, "label", int(group == "positive"))
    table.insert(2, "video_path", folder + "/" + table["file_name"])
    table.insert(3, "video_id", table["file_name"].str.replace(".mp4", "", regex=False))
    tables.append(table)
videos = pd.concat(tables, ignore_index=True)

print("Videos per split (columns: label):")
print(pd.crosstab(videos["split"], videos["label"], margins=True))
videos.head()

Videos per split (columns: label):
label            0     1   All
split                         
test-private   339   338   677
test-public    333   334   667
train          750   750  1500
All           1422  1422  2844


,split,label,video_path,video_id,file_name,time_of_event,time_of_alert,light_conditions,weather,scene,time_to_accident
0,train,1,train/positive/00822.mp4,00822,00822.mp4,19.500,18.633,Normal,Cloudy,Urban,NaN
1,train,1,train/positive/00208.mp4,00208,00208.mp4,19.800,19.233,Normal,Cloudy,Sub-urban,NaN
2,train,1,train/positive/00072.mp4,00072,00072.mp4,20.101,20.068,Normal,Clear,Urban,NaN
3,train,1,train/positive/00128.mp4,00128,00128.mp4,19.267,18.333,Normal,Clear,Urban,NaN
4,train,1,train/positive/00205.mp4,00205,00205.mp4,21.146,18.196,Normal,Clear,Highway,NaN


In [5]:
print("solution.csv:")
print(pd.read_csv(DATA_DIR / "solution.csv").head(), "\n")
print("time_to_accident_test_map.csv:")
print(pd.read_csv(DATA_DIR / "time_to_accident_test_map.csv").head())

solution.csv:
    id  target   Usage  group
0  204       1  Public      0
1   30       1  Public      0
2  146       1  Public      0
3   20       1  Public      0
4  511       1  Public      0 

time_to_accident_test_map.csv:
   0.5     1.0     1.5
0    1  2286.0  2657.0
1    2  2378.0     NaN
2    9  2282.0  2804.0
3   11  2508.0  2810.0
4   12     NaN     NaN


## Metadata Checks

<div dir="rtl">

قبل از استفاده از برچسب‌های زمانی، درستی‌شان بررسی می‌شود: مقدار گمشده، شناسه‌ی تکراری و ترتیب هشدار و حادثه. ستون‌های شرایط محیطی فقط برای تحلیل‌اند و وارد مدل نمی‌شوند، پس مقدار گمشده‌شان با `Unknown` پر می‌شود.

</div>

In [6]:
check_cols = ["time_of_event", "time_of_alert", "time_to_accident",
              "light_conditions", "weather", "scene"]
print("Missing values by split and label:")
print(videos[check_cols].isna().groupby([videos["split"], videos["label"]]).sum())

print("\nDuplicate video ids:", videos["video_id"].duplicated().sum())

pos_train = videos[(videos["split"] == "train") & (videos["label"] == 1)].copy()
pos_train["gap"] = pos_train["time_of_event"] - pos_train["time_of_alert"]
print("Train positives with alert after event:", (pos_train["gap"] < 0).sum())
print("Train positives with gap under 0.1 s:", (pos_train["gap"] < 0.1).sum())

env_cols = ["light_conditions", "weather", "scene"]
videos[env_cols] = videos[env_cols].fillna("Unknown")

print("\nTest clips by time_to_accident:")
test = videos[videos["split"] != "train"]
print(pd.crosstab([test["split"], test["label"]], test["time_to_accident"]))

Missing values by split and label:
                    time_of_event  time_of_alert  time_to_accident  \
split        label                                                   
test-private 0                339            339                 0   
             1                  0              0                 0   
test-public  0                333            333                 0   
             1                  0              0                 0   
train        0                750            750               750   
             1                  0              0               750   

                    light_conditions  weather  scene  
split        label                                    
test-private 0                     0        0      0  
             1                     0        0      0  
test-public  0                     0        0      0  
             1                     0        0      0  
train        0                     0        1      0  
             1     

<div dir="rtl">

**نتیجه.**
- ۲۸۴۴ ویدئو: ۱۵۰۰ ویدئوی train (۷۵۰ مثبت، ۷۵۰ منفی)، ۶۶۷ کلیپ test-public (۳۳۴ مثبت، ۳۳۳ منفی) و ۶۷۷ کلیپ test-private (۳۳۸ مثبت، ۳۳۹ منفی).
- شناسه‌ی تکراری وجود ندارد و در هیچ ویدئوی تصادفی هشدار بعد از حادثه نیامده. زمان حادثه و هشدار فقط در ویدئوهای منفی خالی است و `time_to_accident` فقط در train، که هر دو طبیعی است. تنها مقدار گمشده‌ی دیگر، `weather` در دو ویدئوی train است.
- در ۱۰ ویدئو فاصله‌ی هشدار تا حادثه کمتر از ۰.۱ ثانیه است. یعنی در این ویدئوها خطر تا لحظه‌ی برخورد برای برچسب‌زن‌ها هم دیده نمی‌شده.
- در آزمون، تعداد کلیپ‌های منفی هر گروه Δ تقریباً برابر مثبت‌هاست. مثلاً در test-public، ۱۴۲، ۱۱۵ و ۷۷ کلیپ مثبت در برابر ۱۴۲، ۱۱۶ و ۷۵ کلیپ منفی. پس AP تصادفی در هر گروه حدود ۰.۵ است.

</div>

## Event Timing

<div dir="rtl">

این دو توزیع طراحی برچسب‌ها را تعیین می‌کنند. فاصله‌ی هشدار تا حادثه نشان می‌دهد پنجره‌های «خطر» چقدر جا دارند. زمان حادثه در ویدئو نشان می‌دهد پنجره‌های ویدئوهای عادی باید از کجا بریده شوند تا مدل «جای برش در ویدئو» را یاد نگیرد.

</div>

In [7]:
gap = pos_train["gap"]
print("Alert-to-event gap (s):")
print(gap.describe().round(2))
for delta in [0.5, 1.0, 1.5, 2.0]:
    print(f"Share of positives with gap > {delta} s: {(gap > delta).mean():.1%}")
print("\nTime of event (s):")
print(pos_train["time_of_event"].describe().round(2))

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Time of event in the video (s)", "Alert-to-event gap (s)"])
fig.add_trace(go.Histogram(x=pos_train["time_of_event"], nbinsx=40), row=1, col=1)
fig.add_trace(go.Histogram(x=gap, nbinsx=40), row=1, col=2)
fig.update_layout(height=380, showlegend=False, title="Train positives: event timing")
fig.show()

Alert-to-event gap (s):
count    750.00
mean       1.60
std        0.87
min        0.03
25%        0.99
50%        1.43
75%        2.09
max        4.47
Name: gap, dtype: float64
Share of positives with gap > 0.5 s: 93.3%
Share of positives with gap > 1.0 s: 74.1%
Share of positives with gap > 1.5 s: 46.3%
Share of positives with gap > 2.0 s: 26.7%

Time of event (s):
count    750.00
mean      19.10
std        3.57
min        3.03
25%       19.13
50%       19.80
75%       20.33
max       56.80
Name: time_of_event, dtype: float64


<div dir="rtl">

**نتیجه.**
- فاصله‌ی هشدار تا حادثه به‌طور میانگین ۱.۶۰ ثانیه است (میانه ۱.۴۳، بیشینه ۴.۴۷).
- این فاصله در ۹۳.۳٪ ویدئوها بیشتر از ۰.۵ ثانیه، در ۷۴.۱٪ بیشتر از ۱ ثانیه و در ۴۶.۳٪ بیشتر از ۱.۵ ثانیه است. به همین دلیل گروه ۱.۵ ثانیه در آزمون رسمی کوچک‌تر است.
- حادثه‌ها بیشتر نزدیک ثانیه‌ی ۲۰ رخ می‌دهند (میانه ۱۹.۸۰، نیمه‌ی میانی از ۱۹.۱۳ تا ۲۰.۳۳). پس ویدئوها حول حادثه بریده شده‌اند. اگر پنجره‌های منفی از جای دیگری از ویدئو بریده شوند، «جای برش» می‌تواند به مدل سرنخ بدهد.

</div>

## Conditions

<div dir="rtl">

سهم ویدئوهای تصادف در هر صحنه، آب‌وهوا و نور. اگر این سهم‌ها خیلی فرق کنند، مدلی که کل ویدئو را می‌بیند می‌تواند به‌جای خطر، «نوع جاده» را یاد بگیرد. به همین دلیل در نوت‌بوک مدل، پنجره‌های منفی سخت از خود ویدئوهای تصادف هم بریده می‌شوند. صحنه و نور در آن پنجره‌ها با پنجره‌های خطر یکی است.

</div>

In [8]:
train = videos[videos["split"] == "train"]
fig = make_subplots(rows=1, cols=3, subplot_titles=env_cols)
for i, col in enumerate(env_cols, start=1):
    stats = train.groupby(col)["label"].agg(["mean", "size"]).sort_values("size", ascending=False)
    fig.add_trace(go.Bar(x=stats.index, y=stats["mean"], text=[f"n={n}" for n in stats["size"]],
                         textposition="outside"), row=1, col=i)
    print(f"Collision share by {col}:")
    print(stats.rename(columns={"mean": "positive_share", "size": "videos"}).round(3), "\n")
fig.update_yaxes(range=[0, 1.15])
fig.update_layout(height=400, showlegend=False, title="Share of collision videos by condition (train)")
fig.show()

Collision share by light_conditions:
                  positive_share  videos
light_conditions                        
Normal                     0.502    1355
Twilight                   0.466      88
Dark                       0.468      47
Bright                     0.700      10 

Collision share by weather:
         positive_share  videos
weather                        
Clear             0.484     919
Cloudy            0.491     495
Rain              0.723      83
Unknown           0.500       2
Snow              1.000       1 

Collision share by scene:
            positive_share  videos
scene                             
Urban                0.606     784
Highway              0.451     379
Sub-urban            0.277     271
Other                0.647      34
Rural                0.167      18
Industrial           0.231      13
Nature               1.000       1 



<div dir="rtl">

**نتیجه.**
- سهم تصادف بین صحنه‌ها خیلی فرق می‌کند: Urban ۰.۶۱ (۷۸۴ ویدئو)، Highway ۰.۴۵، Sub-urban ۰.۲۸ و Rural ۰.۱۷. در هوای Rain این سهم ۰.۷۲ است.
- پس مدلی که کل ویدئو را می‌بیند می‌تواند بخشی از دقتش را فقط از نوع صحنه بگیرد.
- گروه‌های Dark (۴۷ ویدئو)، Rain (۸۳) و Rural (۱۸) کوچک‌اند و نتیجه‌ی جداگانه‌ی آن‌ها عدم قطعیت زیادی دارد.

</div>

## Frame Sampling

<div dir="rtl">

از هر ویدئو ۱۰ فریم در ثانیه با اندازه‌ی 392×224 برداشته می‌شود. فریم‌های پشت‌سرهم ۳۰fps تقریباً یکسان‌اند، و گام ۰.۱ ثانیه از گام‌های ارزیابی (۰.۵ ثانیه) خیلی ریزتر است. اندازه‌ی 392×224 نسبت 16:9 را تقریباً نگه می‌دارد و مضرب ۱۴ (اندازه‌ی patch در DINOv2) است. ویدئوهای تصادف train فقط تا ۲ ثانیه بعد از حادثه خوانده می‌شوند، چون فریم‌های بعد از آن برای پیش‌بینی لازم نیستند.

</div>

In [9]:
def read_frames(path, stop_time=None):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps > 0:
        fps = 30.0
    step = fps / FPS_OUT
    frames, times = [], []
    index, next_index = 0, 0.0
    while cap.grab():
        t = index / fps
        if stop_time is not None and t > stop_time:
            break
        if index >= round(next_index):
            ok, frame = cap.retrieve()
            if ok:
                frame = cv2.resize(frame, (FRAME_W, FRAME_H), interpolation=cv2.INTER_AREA)
                frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                times.append(t)
            next_index += step
        index += 1
    cap.release()
    return np.stack(frames), np.array(times, dtype=np.float32), fps, index

In [10]:
demo = pos_train[(pos_train["gap"] >= 1.5) & (pos_train["time_of_event"] >= 4)].sort_values("video_id").iloc[0]
download([demo["video_path"]])
demo_frames, demo_times, demo_fps, _ = read_frames(LOCAL_DIR / demo["video_path"])
print(f"Demo video: {demo['video_path']} | source fps: {demo_fps:.2f} | kept frames: {len(demo_frames)}")
print(f"Frame times: {demo_times[0]:.2f}s to {demo_times[-1]:.2f}s | step: {np.diff(demo_times).mean():.3f}s")
print(f"time_of_alert: {demo['time_of_alert']:.2f}s | time_of_event: {demo['time_of_event']:.2f}s")

offsets = [-3.0, -2.0, -1.5, -1.0, -0.5, 0.0]
picks = [int(np.abs(demo_times - (demo["time_of_event"] + o)).argmin()) for o in offsets]
fig = px.imshow(demo_frames[picks], facet_col=0, facet_col_wrap=3, binary_string=True)
for annotation in fig.layout.annotations:
    i = int(annotation.text.split("=")[1])
    annotation.text = f"event {offsets[i]:+.1f} s"
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(height=430, title=f"Frames before the event (video {demo['video_id']})")
fig.show()

Demo video: train/positive/00000.mp4 | source fps: 28.90 | kept frames: 401
Frame times: 0.00s to 40.00s | step: 0.100s
time_of_alert: 19.14s | time_of_event: 20.76s


<div dir="rtl">

**نتیجه.** از ویدئوی نمونه (۴۰ ثانیه) ۴۰۱ فریم برداشته شد و فاصله‌ی فریم‌ها به‌طور میانگین ۰.۱ ثانیه است. نرخ فریم این ویدئو ۲۸.۹ است، نه ۳۰. برای همین زمان هر فریم از شماره‌ی فریم تقسیم بر fps همان ویدئو حساب می‌شود، نه با فرض ۳۰ فریم در ثانیه.

</div>

## Frame Encoder

<div dir="rtl">

هر فریم با DINOv2-small به بردار تبدیل می‌شود. DINOv2-small یک Vision Transformer با ۲۲ میلیون پارامتر است که بدون برچسب روی ۱۴۲ میلیون تصویر آموزش دیده. وزن‌ها یخ‌زده‌اند، چون fine-tune با ۷۵۰ ویدئوی مثبت overfit می‌شود و روی کولب هم گران است.

از هر فریم ۵ بردار ۳۸۴تایی نگه داشته می‌شود: توکن CLS برای کل تصویر، و میانگین patchها در چهار ربع تصویر تا معلوم باشد هر چیز کجای تصویر است. برش مرکزی پیش‌فرض DINOv2 انجام نمی‌شود تا خودروهای کناری از دست نروند.

</div>

In [11]:
dtype = torch.float16 if device == "cuda" else torch.float32
encoder = AutoModel.from_pretrained(ENCODER_NAME).to(device=device, dtype=dtype).eval()
GRID_H, GRID_W = FRAME_H // 14, FRAME_W // 14
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
print(f"Encoder parameters: {sum(p.numel() for p in encoder.parameters()):,}")
print("Patch grid (rows x columns):", GRID_H, "x", GRID_W)


@torch.inference_mode()
def encode(frames, batch_size=128):
    parts = []
    for i in range(0, len(frames), batch_size):
        x = torch.from_numpy(frames[i:i + batch_size]).to(device)
        x = ((x.permute(0, 3, 1, 2).float() / 255 - MEAN) / STD).to(dtype)
        tokens = encoder(pixel_values=x).last_hidden_state
        grid = tokens[:, 1:].reshape(len(x), GRID_H, GRID_W, -1)
        h, w = GRID_H // 2, GRID_W // 2
        quarters = [grid[:, :h, :w], grid[:, :h, w:], grid[:, h:, :w], grid[:, h:, w:]]
        vectors = [tokens[:, 0]] + [q.mean(dim=(1, 2)) for q in quarters]
        parts.append(torch.stack(vectors, dim=1).float().cpu().numpy())
    return np.concatenate(parts).astype(np.float16)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Encoder parameters: 22,056,576
Patch grid (rows x columns): 16 x 28


In [12]:
started = time.time()
demo_features = encode(demo_frames)
print(f"Demo features: {demo_features.shape} (frames, vectors, size) in {time.time() - started:.1f}s")

cls = demo_features[:, 0].astype(np.float32)
cls /= np.linalg.norm(cls, axis=1, keepdims=True)
change = 1 - (cls[1:] * cls[:-1]).sum(axis=1)
fig = px.line(x=demo_times[1:], y=change, labels={"x": "time (s)", "y": "1 - cosine similarity"},
              title="Frame-to-frame change of the CLS vector (demo video)")
fig.add_vline(x=demo["time_of_alert"], line_dash="dash", line_color="orange", annotation_text="alert")
fig.add_vline(x=demo["time_of_event"], line_dash="dash", line_color="red", annotation_text="event")
fig.show()

Demo features: (401, 5, 384) (frames, vectors, size) in 2.3s


<div dir="rtl">

**نتیجه.**
- ساخت ویژگی ۴۰۱ فریم روی T4 حدود ۲ ثانیه طول کشید.
- در نمودار، تغییر بردار CLS بین هشدار و حادثه چند برابر ثانیه‌های آرام قبل از آن است.
- بعد از حادثه هم تغییر بالا می‌ماند. پس «تغییر زیاد» به‌تنهایی نشانه‌ی خطر نیست و مدل باید الگوی آن را یاد بگیرد.
- این فقط یک ویدئوست و برای نتیجه‌گیری کلی کافی نیست.

</div>

## Feature Extraction

<div dir="rtl">

همه‌ی ویدئوها در تکه‌های ۵۰تایی پردازش می‌شوند. برای هر تکه: دانلود ویدئوها، خواندن فریم‌ها روی CPU با دو thread هم‌زمان، ساخت ویژگی روی GPU، ذخیره‌ی تکه در Drive و پاک کردن ویدئوها. تکه‌هایی که قبلاً ذخیره شده‌اند دوباره پردازش نمی‌شوند، پس اجرای دوباره بعد از قطع کولب از همان‌جا ادامه پیدا می‌کند.

برای داشبورد، از هشت ویدئوی train (چهار تصادف و چهار عادی، به انتخاب تصادفی) یک نسخه‌ی کم‌حجم 640×360 هم ذخیره می‌شود.

</div>

In [13]:
def save_sample_video(source, target):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(source), "-vf", "scale=640:-2",
                    "-c:v", "libx264", "-preset", "veryfast", "-crf", "28", "-an", str(target)],
                   check=True)


sample_paths = set(train.groupby("label").sample(4, random_state=SEED)["video_path"])
print("Dashboard sample videos:", sorted(sample_paths))


def extract_folder(folder):
    rows = videos[videos["video_path"].str.startswith(folder + "/")]
    out_dir = FEATURE_DIR / folder.replace("/", "_")
    out_dir.mkdir(parents=True, exist_ok=True)
    n_chunks = int(np.ceil(len(rows) / CHUNK_SIZE))
    done = sum((out_dir / f"chunk_{k:02d}.npz").exists() for k in range(n_chunks))
    print(f"{folder}: {len(rows)} videos | {n_chunks} chunks | already saved: {done}")
    failed = []
    for k in range(n_chunks):
        out_path = out_dir / f"chunk_{k:02d}.npz"
        if out_path.exists():
            continue
        started = time.time()
        chunk = rows.iloc[k * CHUNK_SIZE:(k + 1) * CHUNK_SIZE]
        download(chunk["video_path"].tolist())
        stops = [r.time_of_event + POST_EVENT if r.split == "train" and r.label == 1 else None
                 for r in chunk.itertuples()]

        feats, times, offsets, paths, fps_list, read_list = [], [], [0], [], [], []
        with ThreadPoolExecutor(max_workers=2) as pool:
            jobs = [pool.submit(read_frames, LOCAL_DIR / p, s) for p, s in zip(chunk["video_path"], stops)]
            for i, video_path in enumerate(chunk["video_path"]):
                try:
                    frames, frame_times, fps, frames_read = jobs[i].result()
                except Exception as error:
                    failed.append((video_path, str(error)))
                    continue
                finally:
                    jobs[i] = None  # release decoded frames
                feats.append(encode(frames))
                times.append(frame_times)
                offsets.append(offsets[-1] + len(frames))
                paths.append(video_path)
                fps_list.append(fps)
                read_list.append(frames_read)
                if video_path in sample_paths:
                    save_sample_video(LOCAL_DIR / video_path, SAMPLE_DIR / Path(video_path).name)
        for video_path in chunk["video_path"]:
            (LOCAL_DIR / video_path).unlink(missing_ok=True)

        if not paths:
            print(f"{folder} | chunk {k + 1}/{n_chunks} | no readable videos")
            continue
        local_file = LOCAL_DIR / f"chunk_{k:02d}.npz"
        np.savez(local_file, features=np.concatenate(feats), times=np.concatenate(times),
                 offsets=np.array(offsets), video_paths=np.array(paths),
                 fps=np.array(fps_list), frames_read=np.array(read_list))
        shutil.copy(local_file, out_path.with_suffix(".tmp"))
        out_path.with_suffix(".tmp").rename(out_path)
        local_file.unlink()
        print(f"{folder} | chunk {k + 1}/{n_chunks} | {len(paths)} videos | "
              f"{sum(len(t) for t in times)} frames | {time.time() - started:.0f}s")
    return failed

Dashboard sample videos: ['train/negative/01071.mp4', 'train/negative/01372.mp4', 'train/negative/01849.mp4', 'train/negative/01875.mp4', 'train/positive/00227.mp4', 'train/positive/00713.mp4', 'train/positive/00754.mp4', 'train/positive/00872.mp4']


In [14]:
failed = []
started = time.time()
for folder in FOLDERS:
    failed += extract_folder(folder)
    print(f"Finished {folder} | elapsed: {(time.time() - started) / 60:.1f} min\n")

print("Unreadable videos:", len(failed))
for video_path, error in failed:
    print(" ", video_path, "|", error)

train/positive: 750 videos | 15 chunks | already saved: 15
Finished train/positive | elapsed: 0.0 min

train/negative: 750 videos | 15 chunks | already saved: 14
train/negative | chunk 15/15 | 50 videos | 18396 frames | 360s
Finished train/negative | elapsed: 6.0 min

test-public/positive: 334 videos | 7 chunks | already saved: 0
test-public/positive | chunk 1/7 | 50 videos | 4989 frames | 92s
test-public/positive | chunk 2/7 | 50 videos | 4974 frames | 92s
test-public/positive | chunk 3/7 | 50 videos | 5016 frames | 101s
test-public/positive | chunk 4/7 | 50 videos | 4979 frames | 95s
test-public/positive | chunk 5/7 | 50 videos | 4994 frames | 97s
test-public/positive | chunk 6/7 | 50 videos | 4987 frames | 97s
test-public/positive | chunk 7/7 | 34 videos | 3377 frames | 61s
Finished test-public/positive | elapsed: 16.6 min

test-public/negative: 333 videos | 7 chunks | already saved: 0
test-public/negative | chunk 1/7 | 50 videos | 4928 frames | 92s
test-public/negative | chunk 2/7 

<div dir="rtl">

**نتیجه.**
- استخراج در دو نوبت کولب انجام شد. نوبت دوم تکه‌های ذخیره‌شده را رد کرد و ۲۹ تکه‌ی باقی‌مانده را در حدود ۴۹ دقیقه ساخت.
- هیچ ویدئویی ناخوانا نبود.
- هشدار کمبود جا در سلول Setup (۱.۸ گیگ آزاد) مربوط به همین نوبت دوم است، چون ویژگی‌های نوبت اول از قبل در Drive بودند.
- هر تکه‌ی ۵۰تایی از ویدئوهای عادی (۴۰ ثانیه‌ای) حدود ۶ دقیقه و هر تکه از کلیپ‌های آزمون (۱۰ ثانیه‌ای) حدود ۱.۵ دقیقه طول کشید.

</div>

## Output Checks

<div dir="rtl">

همه‌ی تکه‌ها دوباره خوانده می‌شوند و برای هر ویدئو بررسی می‌شود که ویژگی دارد، مقدار نامعتبر ندارد و زمان فریم‌هایش صعودی است. در ویدئوهای تصادف train هم بررسی می‌شود که فریم‌ها از لحظه‌ی حادثه گذشته باشند. مشخصات فریم‌های هر ویدئو به جدول متادیتا اضافه و در `videos.csv` ذخیره می‌شود.

</div>

In [15]:
records, problems = [], []
for folder in FOLDERS:
    for path in sorted((FEATURE_DIR / folder.replace("/", "_")).glob("chunk_*.npz")):
        with np.load(path) as chunk:
            feats, times, offsets = chunk["features"], chunk["times"], chunk["offsets"]
            for i, video_path in enumerate(chunk["video_paths"]):
                f = feats[offsets[i]:offsets[i + 1]]
                t = times[offsets[i]:offsets[i + 1]]
                if not np.isfinite(f).all() or not (np.diff(t) > 0).all():
                    problems.append(str(video_path))
                records.append({"video_path": str(video_path), "n_frames": len(t),
                                "first_time": float(t[0]), "last_time": float(t[-1]),
                                "source_fps": float(chunk["fps"][i]),
                                "feature_file": path.relative_to(FEATURE_DIR).as_posix()})

videos_out = videos.merge(pd.DataFrame(records), on="video_path", how="left")
print("Videos with features:", videos_out["n_frames"].notna().sum(), "of", len(videos_out))
print("Videos with invalid values or non-increasing times:", len(problems))
pos = videos_out[(videos_out["split"] == "train") & (videos_out["label"] == 1)]
print("Train positives whose frames stop before the event:", (pos["last_time"] < pos["time_of_event"]).sum())
print("Source fps values:", videos_out["source_fps"].round(2).value_counts().to_dict())
print("\nKept frames per video:")
print(videos_out.groupby(["split", "label"])["n_frames"].describe().round(1))

videos_out.to_csv(DATA_DIR / "videos.csv", index=False)

Videos with features: 2844 of 2844
Videos with invalid values or non-increasing times: 0
Train positives whose frames stop before the event: 0
Source fps values: {30.0: 1692, 30.6: 445, 29.9: 207, 30.5: 160, 29.8: 39, 29.2: 31, 29.7: 25, 29.4: 25, 29.3: 24, 30.4: 22, 29.6: 22, 29.5: 18, 29.1: 17, 28.9: 16, 30.3: 13, 31.0: 10, 28.7: 9, 30.2: 9, 29.0: 8, 30.1: 6, 28.0: 6, 28.4: 4, 22.8: 3, 27.4: 3, 26.8: 3, 27.5: 3, 25.2: 3, 26.4: 2, 24.8: 2, 22.2: 2, 27.1: 2, 26.7: 1, 28.1: 1, 27.7: 1, 28.5: 1, 28.3: 1, 25.6: 1, 25.3: 1, 28.6: 1, 24.6: 1, 23.6: 1, 23.9: 1, 24.1: 1, 27.3: 1}

Kept frames per video:
                    count   mean   std    min    25%    50%    75%    max
split        label                                                       
test-private 0      339.0   98.5   6.5   66.0   98.0  100.0  102.0  108.0
             1      338.0  100.4   3.5   86.0   98.0  100.0  103.0  110.0
test-public  0      333.0   97.5   7.3   74.0   97.0   99.0  102.0  114.0
             1      334.0 

<div dir="rtl">

**نتیجه.**
- هر ۲۸۴۴ ویدئو ویژگی دارند. مقدار نامعتبر یا زمان غیرصعودی پیدا نشد، و در همه‌ی ویدئوهای تصادف فریم‌ها از لحظه‌ی حادثه گذشته‌اند.
- نرخ فریم ۱۶۹۲ ویدئو ۳۰ است و بقیه بین ۲۲.۲ و ۳۱.۰ هستند. پس حساب کردن زمان هر فریم از fps خودِ همان ویدئو لازم بود.
- تعداد فریم‌ها (میانه):
  - ویدئوهای عادی train: ۴۰۱ فریم (۴۰ ثانیه)
  - ویدئوهای تصادف: ۲۱۹ فریم، چون تا ۲ ثانیه بعد از حادثه خوانده شده‌اند
  - کلیپ‌های آزمون: حدود ۱۰۰ فریم (۱۰ ثانیه)
- حجم ویژگی‌ها در Drive ۲.۲ گیگ است، در برابر ۳۱ گیگ کل دیتاست.
- زمان هر فریم از شماره‌ی فریم و fps میانگین ویدئو حساب شده است. این فقط وقتی درست است که نرخ فریم در طول ویدئو ثابت باشد. در نوت‌بوک `anticipation_frame_check` این فرض روی ۴۰ ویدئوی تصادف بررسی شد: خطای زمان صفر بود و همه‌ی ویدئوها، حتی آن‌هایی که fps آن‌ها ۳۰ نیست، نرخ فریم ثابت داشتند.

</div>

## Saved Files

<div dir="rtl">

حجم خروجی‌های ذخیره‌شده در Drive.

</div>

In [17]:
for folder in [DATA_DIR, FEATURE_DIR, SAMPLE_DIR]:
    files = [p for p in folder.rglob("*") if p.is_file()]
    size = sum(p.stat().st_size for p in files) / 1e6
    print(f"{folder.relative_to(DRIVE_DIR)}: {len(files)} files | {size:,.0f} MB")

data: 6 files | 0 MB
features: 58 files | 2,193 MB
sample_videos: 8 files | 12 MB
